In [1]:
import re
from collections import Counter

# Q2.3.1 Text used to train the BPE model
training_text = (
    "Tell your heart that the fear of suffering is worse than "
    "the suffering itself. "
    "And that no heart has ever suffered when it goes in search "
    "of its dreams, because every second of the search is a "
    "second's encounter with God and with eternity."
)


def word_frequency(text):
    # Extract words and count how often each one appears.
    extracted_words = re.findall(
        r"[A-Za-z]+(?:'[A-Za-z]+)?",
        text.lower()
    )

    return Counter(extracted_words)


def character_tokens(word):
    # Represent a word as characters followed by an end marker.
    return list(word) + ["_"]


def pair_frequency(tokenized_words, word_counts):
    # Calculate the frequency of every neighboring token pair.
    pair_counts = Counter()

    for word, frequency in word_counts.items():
        tokens = tokenized_words[word]

        for first, second in zip(tokens, tokens[1:]):
            pair_counts[(first, second)] += frequency

    return pair_counts


def combine_pair(tokenized_words, target_pair):
    # Merge the selected pair throughout the training vocabulary.
    new_tokenized_words = {}

    for word, tokens in tokenized_words.items():
        merged_tokens = []
        index = 0

        while index < len(tokens):
            if (
                index + 1 < len(tokens)
                and (tokens[index], tokens[index + 1])
                == target_pair
            ):
                merged_tokens.append(
                    tokens[index] + tokens[index + 1]
                )
                index += 2
            else:
                merged_tokens.append(tokens[index])
                index += 1

        new_tokenized_words[word] = merged_tokens

    return new_tokenized_words


def build_bpe(text, maximum_merges=30):
    # Learn BPE merges from the supplied paragraph.
    counts = word_frequency(text)

    tokenized_words = {
        word: character_tokens(word)
        for word in counts
    }

    merge_records = []

    for step in range(maximum_merges):
        pair_counts = pair_frequency(
            tokenized_words,
            counts
        )

        if not pair_counts:
            break

        best_pair = min(
            pair_counts,
            key=lambda pair: (
                -pair_counts[pair],
                pair
            )
        )

        best_frequency = pair_counts[best_pair]

        merge_records.append(
            (best_pair, best_frequency)
        )

        tokenized_words = combine_pair(
            tokenized_words,
            best_pair
        )

        print(
            f"Merge {step + 1}: "
            f"{best_pair} -> {best_frequency}"
        )

    return counts, tokenized_words, merge_records


def encode_word(word, merge_records):
    # Segment a new word using the learned merge sequence.
    pieces = character_tokens(word.lower())

    for target_pair, _ in merge_records:
        new_pieces = []
        index = 0

        while index < len(pieces):
            if (
                index + 1 < len(pieces)
                and (pieces[index], pieces[index + 1])
                == target_pair
            ):
                new_pieces.append(
                    pieces[index] + pieces[index + 1]
                )
                index += 2
            else:
                new_pieces.append(pieces[index])
                index += 1

        pieces = new_pieces

    return pieces


if __name__ == "__main__":
    print("Q2.3 BPE Training")
    print("=================")

    counts, final_tokens, merge_records = build_bpe(
        training_text,
        maximum_merges=30
    )

    print("\nFive Most Frequent Merges")
    print("=========================")

    top_merges = sorted(
        merge_records,
        key=lambda item: (
            -item[1],
            item[0]
        )
    )[:5]

    for pair, frequency in top_merges:
        print(
            f"{pair} : {frequency}"
        )

    print("\nFive Longest Resulting Subword Tokens")
    print("=====================================")

    vocabulary = {
        token
        for tokens in final_tokens.values()
        for token in tokens
    }

    longest_tokens = sorted(
        vocabulary,
        key=lambda token: (
            -len(token),
            token
        )
    )[:5]

    for token in longest_tokens:
        print(token)

    print("\nQ2.3.3 Word Segmentation")
    print("========================")

    words_to_test = [
        "suffering",
        "eternity",
        "encounter",
        "dreams",
        "heart"
    ]

    for word in words_to_test:
        print(
            f"{word}: "
            f"{encode_word(word, merge_records)}"
        )

Q2.3 BPE Training
Merge 1: ('t', 'h') -> 8
Merge 2: ('e', 'r') -> 7
Merge 3: ('s', '_') -> 7
Merge 4: ('s', 'e') -> 7
Merge 5: ('a', 'r') -> 5
Merge 6: ('d', '_') -> 5
Merge 7: ('t', '_') -> 5
Merge 8: ('f', '_') -> 4
Merge 9: ('a', 'n') -> 3
Merge 10: ('c', 'o') -> 3
Merge 11: ('e', '_') -> 3
Merge 12: ('e', 'ar') -> 3
Merge 13: ('f', 'er') -> 3
Merge 14: ('f', 'fer') -> 3
Merge 15: ('i', 'n') -> 3
Merge 16: ('i', 't') -> 3
Merge 17: ('o', 'f_') -> 3
Merge 18: ('s', 'u') -> 3
Merge 19: ('su', 'ffer') -> 3
Merge 20: ('th', 'e_') -> 3
Merge 21: ('a', 't_') -> 2
Merge 22: ('an', 'd_') -> 2
Merge 23: ('ar', 'c') -> 2
Merge 24: ('arc', 'h') -> 2
Merge 25: ('arch', '_') -> 2
Merge 26: ('co', 'n') -> 2
Merge 27: ('e', 'n') -> 2
Merge 28: ('e', 'v') -> 2
Merge 29: ('ear', 't_') -> 2
Merge 30: ('er', '_') -> 2

Five Most Frequent Merges
('t', 'h') : 8
('e', 'r') : 7
('s', '_') : 7
('s', 'e') : 7
('a', 'r') : 5

Five Longest Resulting Subword Tokens
suffer
arch_
eart_
and_
the_

Q2.3.3 Word Seg